# Create Combined Scopus17 Dataset

This notebook merges the raw per-SDG Scopus exports into a multilabel corpus, cleans and deduplicates abstracts, and writes combined train, validation, and test CSV and Parquet datasets.


In [ ]:
%pip install levenshtein

In [ ]:
import time
import urllib
from pathlib import Path
from typing import Dict, List

import jupyter_black
import Levenshtein as lev
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import regex as re
import requests
import seaborn as sns
from dotenv import load_dotenv
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from tqdm.auto import tqdm
from transformers import BertTokenizer

# Use verbose pandas mode
pd.options.display.max_rows = 200

load_dotenv()
jupyter_black.load()

In [ ]:
data_path = Path("data")

for i in range(1, 18):  # Loop through SDG 1 to 17
    file_path = data_path / "raw" / "scopus" / f"SDG{i:02}.csv"  # Constructs file path
    try:
        df = pd.read_csv(file_path)

        row_count = len(df)

        if row_count != 20000:
            print(f"Warning: {file_path} does not have 20,000 rows.")
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

In [ ]:
def clean_abstract(abstract):
    df = pd.DataFrame(columns=["Abstract"])
    df["Abstract"] = [abstract]

    copyright_regex = r"^((?:Copyright ©|©) (?:\d{4})?(?:[[:alpha:][:punct:]\s][^\.]+?)[A-Za-z)]([A-Z][a-z\s\d[:punct:]]))"

    df["Abstract"] = df["Abstract"].apply(
        lambda abstract: re.sub(
            copyright_regex, lambda match: match.group(2), abstract
        ).strip()
    )

    return df["Abstract"].iloc[0]


# Test the function
sample_abstract = "© 2023 Chongqing Medical UniversityB-cell CLL/lymphoma 9 (BCL9) is considered a key developmental"
cleaned_abstract = clean_abstract(sample_abstract)
print(cleaned_abstract)

In [ ]:
def clean_abstracts(df):
    df_cleaned = df.copy()

    df_cleaned["Abstract"] = df_cleaned["Abstract"].str.strip()

    # for _, row in df_cleaned.iterrows():
    #     if "Success in marriage markets has lasting" in row["Abstract"]:
    #         print(f"Found it: {row['Abstract']}")

    # First, the specific case where there is no punctuation or space between the year and the first letter of the abstract
    copyright_regex = r"(© \d{4})([A-Z])"

    df_cleaned["Abstract"] = df_cleaned["Abstract"].apply(
        lambda abstract: re.sub(
            copyright_regex, lambda match: match.group(2), abstract
        ).strip()
    )

    # Next, handle cases where the end of the copyright statement and the beginning of the next sentence are merged without a space.
    copyright_regex = r"^((?:Copyright ©|©) (?:\d{4})?(?:[[:alpha:][:punct:]\s][^\.]+?)[A-Za-z)]([A-Z][a-z\s\d[:punct:]]))"

    df_cleaned["Abstract"] = df_cleaned["Abstract"].apply(
        lambda abstract: re.sub(
            copyright_regex, lambda match: match.group(2), abstract
        ).strip()
    )

    # Then, remove specific strings that are known to be present in the abstracts
    copyright_regexes = [
        r"^(?:Copyright ©|©) (\d{4}) Elsevier B\.V\.",
        r"^© Elsevier B\.V\.",
        r"© \d{4} Elsevier Ltd",
        r"© \d{4} Published by Elsevier Ltd\.",
        r"Published by Elsevier Ltd\.",
        r"S\. Government work and not under copyright protection in the US; foreign copyright protection may apply\.",
        r"^© The Author(s) \d{4}\.",
        r"^© \d{4} The author\(s\)\.",
        r"^© \d{4} The authors\.",
        r"^© \d{4} The author\.",
        r"^© \d{4}, The Author\(s\)\.",
        r"^© \d{4} The Author\(s\)",
        r"^© \d{4} The Author",
        r"^© \d{4} The Authors",
        r"\(Figure presented\.\)© The Author\(s\) \d{4}\.",
        r"ICIC International © \d{4}\.",
        r"^©2024 National Information and Documentation Center \(NIDOC\)",
        r"^© E. Green and F. Ritchie.",
        r"^@ \d{4} China University of Geosciences \(Beijing\) and Peking University",
        r"^© \d{4} Akademikerförbundet SSR \(ASSR\) and John Wiley \& Sons Ltd\.",
        r"© \d{4} Society of Chemical Industry\.",
        r"^© by the author, licensee University of Lodz – Lodz University Press, Lodz, Poland\.",
        r"This is an Open Access article under the CC BY(?:-NC-ND|-SA)? 4\.0 license\.?",
        r"This is an open access article under the CC BY-SA license\.?",
        r"This is an open access article under the CC BY-NC License \(http://creativecommons.org/licenses/by-nc/4\.0/\)\.",
        r"This is an open access article under the CC BY-SA https://creativecommons.org/licenses/by-sa/4\.0/\.",
        r"© This is an open access article under the CC BY license (http://creativecommons.org/licenses/by/4\.0/)\.",
        r"This is an Open Access article distributed under the terms of the Creative Commons Attribution License, which permits unrestricted use, distribution, and reproduction in any medium, provided the original work is properly cited\.",
        r"This is an open access article distributed under the terms of the Creative Commons Attribution 4\.0 International License \(https://creativecommons.org/licenses/by/4\.0/\), allowing third parties to copy and redistribute the material in any medium or format and to remix, transform, and build upon the material for any purpose, even commercially, provided the original work is properly cited and states its license\.",
        r"Published by Wolters Kluwer Health, Inc\.",
        r"Journal of The Science of Food and Agriculture published by John Wiley \& Sons Ltd on behalf of Society of Chemical Industry\.",
        r"International Transactions in Operational Research © 2022 International Federation of Operational Research Societies\.",
        r"(^©.*?All rights reserved\.?)",
        r"(^Copyright.*?All rights reserved\.?)",
        r"^© \d{4} Copyright:",
        r"© \d{4} American Medical Association\.",
        r"©\d{4} American Medical Association\.",
        r"© \d{4} AMA\.",
        r"© \d{4} \w+ \w+ \w+\.",
        r"© \d{4}, \w+ \w+ \w+\.",
        r"© \d{4} \w+ \w+ \w+ \w+\. All rights reserved."
        r"© \d{4} \w+ \w+\."
        r"All rights reserved\.",
        r"© \d{4}, \w+ \w+\." r"All rights reserved\.",
        r"© \d{4} American Medical Association All rights reserved\."
        r"Copyright \d{4} American Medical Association\. All rights reserved\.",
        r"Copyright © \d{4} JAMA - Journal of the American Medical Association\. All rights reserved\.",
        r"© \d{4} JAMA - Journal of the American Medical Association\. All rights reserved\.",
        r"© JAMA - Journal of the American Medical Association \d{4}\.",
        r"© \d{4} National Academy of Sciences\. All rights reserved\.",
        r"© \d{4} The Authors",
        r"© The Author\(s\) \d{4}\.",
        r"© \d{4} WILEY-VCH Verlag GmbH & Co\. KGaA, Weinheim",
        r"© \d{4}, Published with license by Taylor & Francis\.",
        r"© \d{4} Financial Management Association International",
        r"\[copyright information to be updated in production process\]\.",
    ]

    for regex in copyright_regexes:
        df_cleaned["Abstract"] = df_cleaned["Abstract"].apply(
            lambda x: re.sub(regex, "", x, flags=re.IGNORECASE).strip()
        )

    # Next, handle other weird cases.
    copyright_regex = r"^© 2023‘"

    df_cleaned["Abstract"] = df_cleaned["Abstract"].str.replace(
        copyright_regex, r"‘", regex=True
    )

    copyright_regex = r"^© 20231\-MeV"

    df_cleaned["Abstract"] = df_cleaned["Abstract"].str.replace(
        copyright_regex, r"1-MeV", regex=True
    )

    copyright_regex = r"^© 20233D"

    df_cleaned["Abstract"] = df_cleaned["Abstract"].str.replace(
        copyright_regex, r"3D", regex=True
    )

    # Do these last.
    copyright_regexes = [
        r"^©\s?(?:\d{4})?.*?\.",
        r"(^Copyright\s*©.*?\.)",
        r"(^Copyright\s*:\s*©.*?\.)",
        r"© \d{4} American Society of Civil Engineers\.",
        r"Global Change Biology© \d{4} The Authors\.",
        r"© \d{4},? The Author\(s\)\.",
        r"© \d{4},? The Authors.",
    ]

    for regex in copyright_regexes:
        df_cleaned["Abstract"] = df_cleaned["Abstract"].apply(
            lambda x: re.sub(regex, "", x, flags=re.IGNORECASE).strip()
        )

    # Optionally, strip any leading/trailing whitespace that might be left after the replacement
    df_cleaned["Abstract"] = df_cleaned["Abstract"].str.strip()
    # Reset the index after dropping rows
    df_cleaned.reset_index(drop=True, inplace=True)

    return df_cleaned

In [ ]:
# Read and combine all dataframes; clean the data
combined_df = pd.DataFrame()
for i in range(1, 18):
    file_path = data_path / "raw" / "scopus" / f"SDG{i:02}.csv"  # Constructs file path
    df = pd.read_csv(file_path)

    # Remove rows with no Title or Abstract
    # df = df[df["Abstract"] != "[No abstract available]"].copy()
    # df = df[df["Abstract"] != "None"].copy()
    # df = df[df["Abstract"] != ""].copy()
    # df = df.dropna(subset=['Abstract']).copy()
    # df = df.dropna(subset=['Title']).copy()

    df = df[
        ~df["Abstract"].isin(["[No abstract available]", "None", ""])
        & df["Title"].notna()
        & df["Abstract"].notna()
    ].copy()

    # Reset the index after dropping rows
    df.reset_index(drop=True, inplace=True)

    # Clean abstracts
    df = clean_abstracts(df)

    df["SDG"] = i

    row_count = len(df)
    print(f"SDG {i} - Row count: {row_count}")

    combined_df = pd.concat([combined_df, df])

In [ ]:
count = 0
for abstract in combined_df["Abstract"]:
    if (
        "copyright" in abstract.lower()
        or "©" in abstract
        or "open access article" in abstract.lower()
    ):
        print(abstract)
        count += 1
    if count > 3:
        break

In [ ]:
# Check that the rows we're merging are actually the same
# We're using Levenshtein distance to check for similarity because some of the abstracts have minor differences
# due to formatting, etc. but they are actually the same abstract (e.g. extra whitespace, etc.).
def check_consistency(series, field):
    threshold = 5
    unique_values = series.unique()
    if len(unique_values) > 1:
        # Calculate pairwise Levenshtein distance and check if all are within the threshold
        for i in range(len(unique_values)):
            for j in range(i + 1, len(unique_values)):
                if lev.distance(unique_values[i], unique_values[j]) > threshold:
                    error_message = f"Inconsistency found in field '{field}'. Values '{unique_values[i]}' and '{unique_values[j]}' differ significantly."
                    # raise ValueError(error_message)
                    print(error_message)
    return unique_values[0]


# Group by 'DOI' and 'Title', and aggregate data
try:
    combined_df = (
        combined_df.groupby(["DOI", "Title"])
        .agg(
            {
                "Abstract": lambda x: check_consistency(x, "Abstract"),
                "Author Keywords": lambda x: check_consistency(x, "Author Keywords"),
                "SDG": lambda x: list(set(x)),
            }
        )
        .reset_index()
    )
except ValueError as e:
    print(f"Error: {e}")


# Create binary SDG vectors
def create_sdg_vector(row, total_sdgs=17):
    vector = [0] * total_sdgs
    for sdg in row["SDG"]:
        vector[sdg - 1] = 1
    return vector


combined_df["SDG Labels"] = combined_df.apply(create_sdg_vector, axis=1)

# for i, row in combined_df.iterrows():
#     if len(row['SDG']) > 7:
#         print(row['SDG'])
#         print(row['SDG Labels'])
#         break

print(combined_df.shape)
combined_df.head()

In [ ]:
# # Concatenate 'Title', 'Abstract', and 'Author Keywords' into a single text field
# # Ensuring natural flow of the text
# combined_df["Article Text"] = combined_df["Title"] + ". " + combined_df["Abstract"]

# # Include 'Author Keywords' only if it is not NaN and not 'No author keywords found'
# combined_df["Article Text"] = combined_df.apply(
#     lambda row: (
#         row["Article Text"] + " Keywords: " + row["Author Keywords"]
#         if pd.notna(row["Author Keywords"])
#         and row["Author Keywords"] != "No author keywords found"
#         else row["Article Text"]
#     ),
#     axis=1,
# )

# print(combined_df["Article Text"].iloc[44])
# print(combined_df["Article Text"].iloc[63])
# print(combined_df["Article Text"].iloc[400])

In [ ]:
# Concatenate 'Title', 'Abstract', and 'Author Keywords' into a single text field
# Ensuring natural flow of the text
combined_df["Article Text"] = combined_df["Abstract"]

# Include 'Author Keywords' only if it is not NaN and not 'No author keywords found'
# combined_df['Article Text'] = combined_df.apply(
#     lambda row: row['Article Text'] + " Keywords: " + row['Author Keywords']
#     if pd.notna(row['Author Keywords']) and row['Author Keywords'] != "No author keywords found"
#     else row['Article Text'], axis=1)

print(combined_df["Article Text"].iloc[44])
print(combined_df["Article Text"].iloc[63])
print(combined_df["Article Text"].iloc[400])

In [ ]:
sdg_counts = combined_df["SDG Labels"].apply(pd.Series).sum().astype(int)

# Create the bar plot
sdg_counts.plot(kind="bar")
plt.title("Distribution of Papers Across SDGs")
plt.xlabel("SDG Number")
plt.ylabel("Number of Papers")
plt.xticks(ticks=range(17), labels=range(1, 18), rotation=0)
plt.show()

In [ ]:
# Count the number of SDGs supported per paper
num_sdg_per_paper = combined_df["SDG Labels"].apply(lambda x: sum(x))

# Count the frequency of each sum (number of SDGs supported)
num_sdg_counts = num_sdg_per_paper.value_counts().sort_index()

# Plotting
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
num_sdg_counts.plot(kind="bar")
plt.title("SDGs Supported per Paper")
plt.xlabel("Number of SDGs Supported")
plt.ylabel("Number of Papers")
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
num_sdg_counts.plot(kind="bar", logy=True)
plt.title("SDGs Supported per Paper (Log Scale)")
plt.xlabel("Number of SDGs Supported")
plt.ylabel("Log of Number of Papers")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
combined_df[combined_df["SDG Labels"].apply(lambda x: sum(x) > 8)]

In [ ]:
# loss_mask = [
#     np.full(
#         16,
#         0.5,  # Low confidence for the implicit labels
#         dtype=float,
#     )
#     for _ in range(len(combined_df))
# ]
# for idx, _ in combined_df.iterrows():
#     active_idx = np.where(np.array(combined_df.at[idx, "SDG Labels"]) == 1)[0]
#     loss_mask[idx][active_idx] = 1  # High confidence for the explicit label


def update_loss_mask(row, num_labels, implicit_label_confidence):
    # Create a loss mask with default confidence values
    loss_mask = np.full(num_labels, implicit_label_confidence, dtype=float)
    # Update the mask where labels are 1
    active_idxs = np.where(np.array(row["SDG Labels"]) == 1)[0]
    loss_mask[active_idxs] = 1  # High confidence for the explicit labels
    return loss_mask


# Apply the function to each row
loss_mask = combined_df.apply(
    update_loss_mask, axis=1, num_labels=17, implicit_label_confidence=0.5
).tolist()

loss_mask[9661]

In [ ]:
all_sdgs_supported = combined_df[combined_df["SDG Labels"].apply(lambda x: sum(x) == 7)]
sampled_rows = all_sdgs_supported.sample(n=1)

print(f"DOI: {sampled_rows['DOI'].iloc[0]}\nTitle: {sampled_rows['Title'].iloc[0]}")

In [ ]:
columns_to_keep = ["DOI", "Article Text", "SDG Labels"]
df_cleaned_final = combined_df[columns_to_keep]

In [ ]:
# Assuming df['SDG Labels'] is a numpy array of shape (num_samples, num_labels)
X = df_cleaned_final.drop(columns=["SDG Labels"])  # Features
y = np.stack(df_cleaned_final["SDG Labels"])  # Multi-label targets

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=200)

for train_index, test_index in msss.split(X, y):
    train_df = df_cleaned_final.iloc[train_index]
    test_df = df_cleaned_final.iloc[test_index]

# reset the indices
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
# Assuming df['SDG Labels'] is a numpy array of shape (num_samples, num_labels)
X = train_df.drop(columns=["SDG Labels"])  # Features
y = np.stack(train_df["SDG Labels"])  # Multi-label targets

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=200)

for train_index, val_index in msss.split(X, y):
    new_train_df = train_df.iloc[train_index]
    val_df = train_df.iloc[val_index]

# reset the indices
train_df = new_train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [ ]:
print(f"Train: {train_df.shape}, Validation: {val_df.shape}, Test: {test_df.shape}")

In [ ]:
train_file_path = data_path / "processed" / "scopus17_2023_train.csv"
train_df.to_csv(train_file_path)
val_file_path = data_path / "processed" / "scopus17_2023_val.csv"
val_df.to_csv(val_file_path)
test_file_path = data_path / "processed" / "scopus17_2023_test.csv"
test_df.to_csv(test_file_path)

train_file_path = data_path / "processed" / "scopus17_2023_train.parquet"
train_df.to_parquet(train_file_path)
val_file_path = data_path / "processed" / "scopus17_2023_val.parquet"
val_df.to_parquet(val_file_path)
test_file_path = data_path / "processed" / "scopus17_2023_test.parquet"
test_df.to_parquet(test_file_path)

In [ ]:
# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-cased")

df = df_cleaned_final.copy()

# Tokenize and calculate token lengths
df["Token Length"] = df["Article Text"].apply(lambda x: len(tokenizer.tokenize(x)))

In [ ]:
# Plotting the distribution of token lengths
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, x=df.index, y="Token Length", marker="o"
)  # 'o' is a filled circle
plt.title("Distribution of Token Lengths in Article Text")
plt.xlabel("Article Index")
plt.ylabel("Token Length")
plt.show()

# Calculate median, mean, and standard deviation
median_length = df["Token Length"].median()
mean_length = df["Token Length"].mean()
std_dev_length = df["Token Length"].std()

print(f"Median Token Length: {median_length}")
print(f"Mean Token Length: {mean_length}")
print(f"Standard Deviation of Token Length: {std_dev_length}")